# Solutions · 15. Gradient boosting and the Huber loss

Worked solutions to the three exercises of [Lecture 15](../../notebooks/part5_ensembles/15_functional_gradient_boosting_and_huber_clipping.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Stagewise L2 Boosting on a Synthetic Hardness Curve of a Binary Alloy")
x_Ni = np.linspace(0.0, 1.0, 45)
# Synthetic hardness curve of a binary alloy: a parabolic term plus an asymmetric term (assumed constants, HV):
HV_true = 65.0 + 180.0 * x_Ni * (1.0 - x_Ni) + 35.0 * (x_Ni ** 2)

mses_stage = []
for m_trees in [1, 5, 15, 35]:
    gbm = trees_ensembles.GradientBoostingRegressor(n_estimators=m_trees, learning_rate=0.2, max_depth=2).fit(x_Ni, HV_true)
    mses_stage.append(float(np.mean((gbm.predict(x_Ni) - HV_true) ** 2)))
    print(f"Stage M = {m_trees:2d} shallow trees (depth=2) | Hardness MSE = {mses_stage[-1]:.4f} HV^2")

assert all(mses_stage[i] > mses_stage[i + 1] for i in range(len(mses_stage) - 1)) and mses_stage[-1] < 1.0
heading("Demonstration 2: Unclipped L2 Pseudo-Residuals vs. Huber-Clipped Pseudo-Residuals")
rng = np.random.default_rng(42)
HV_noisy = HV_true + rng.normal(0.0, 1.8, size=len(x_Ni))
HV_spiked = HV_noisy.copy()
HV_spiked[0] += 250.0
HV_spiked[-1] += 250.0

r1_l2 = HV_spiked - float(np.mean(HV_spiked))
r1_huber = np.clip(HV_spiked - float(np.median(HV_spiked)), -5.0, 5.0)
print(f"Stage-1 L2 initial baseline mean(y)      = {np.mean(HV_spiked):.2f} HV | Spike pseudo-residual r_0 = {r1_l2[0]:+.2f} HV")
print(f"Stage-1 Huber initial baseline median(y) = {np.median(HV_spiked):.2f} HV | Clipped pseudo-residual r_0 = {r1_huber[0]:+.2f} HV")
assert abs(r1_l2[0]) > 180.0 and abs(r1_huber[0]) == 5.0
heading("Demonstration 3: L2 GBM vs. Huber-Clipped GBM Benchmark on Outlier-Corrupted Alloy Data")
gbm_l2 = trees_ensembles.GradientBoostingRegressor(n_estimators=25, learning_rate=0.15, max_depth=2).fit(x_Ni, HV_spiked)
gbm_huber = trees_ensembles.HuberGradientBoostingRegressor(n_estimators=25, learning_rate=0.15, max_depth=2, delta=5.0).fit(x_Ni, HV_spiked)

rmse_l2 = float(np.sqrt(np.mean((gbm_l2.predict(x_Ni) - HV_true) ** 2)))
rmse_huber = float(np.sqrt(np.mean((gbm_huber.predict(x_Ni) - HV_true) ** 2)))
print(f"RMSE vs the true curve under 2 gross spikes: L2 GBM = {rmse_l2:.3f} HV -> Huber Clipped GBM = {rmse_huber:.3f} HV")
assert rmse_huber < 0.35 * rmse_l2
heading("Demonstration 4: Shrinkage Learning Rate eta & Huber Clipping Threshold delta Sweep")
print(f"{'Shrinkage eta':>13s} | {'Huber delta [HV]':>16s} | {'RMSE vs True Alloy Curve [HV]':>29s}")
print("-" * 64)
for lr, d_h in [(0.05, 5.0), (0.15, 2.0), (0.15, 5.0), (0.15, 25.0), (0.15, 250.0), (0.40, 5.0)]:
    gb_s = trees_ensembles.HuberGradientBoostingRegressor(n_estimators=30, learning_rate=lr, max_depth=2, delta=d_h).fit(x_Ni, HV_spiked)
    err_s = float(np.sqrt(np.mean((gb_s.predict(x_Ni) - HV_true) ** 2)))
    print(f"{lr:13.2f} | {d_h:16.1f} | {err_s:29.3f}")
heading("Inside the Algorithm: By-Hand Huber Boosting Loop vs. the Library")
f0_hand = float(np.median(HV_spiked))
pred_hand = np.full_like(HV_spiked, f0_hand)
for _ in range(25):
    p_res = np.clip(HV_spiked - pred_hand, -5.0, 5.0)
    t_stage = trees_ensembles.DecisionTreeRegressor(max_depth=2, min_samples_split=6).fit(x_Ni, p_res)
    pred_hand += 0.15 * t_stage.predict(x_Ni)
diff_hgbm = float(np.max(np.abs(pred_hand - gbm_huber.predict(x_Ni))))
print("by hand (pure NumPy) pred[:5]:", np.round(pred_hand[:5], 5))
print("library (engml_2_applied)      pred[:5]:", np.round(gbm_huber.predict(x_Ni)[:5], 5))
print(f"Max discrepancy              : {diff_hgbm:.2e}")
assert diff_hgbm < 1e-14
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** For a 0-split constant stump update $h_m(x) = r_{m-1}$, prove and verify numerically that after $M = 20$ boosting stages with shrinkage $\eta = 0.15$ and initial residual $r_0 = 10.0$, the stage-$M$ residual equals the exact geometric contraction $(1 - \eta)^M r_0$ to machine precision.

In [2]:
heading("Exercise 1: Geometric Residual Contraction r_M = (1 - eta)^M r_0 Verification")
eta, M = 0.15, 20
r0 = 10.0
r_iter = r0
for _ in range(M):
    r_iter = r_iter - eta * r_iter
r_exact = ((1.0 - eta) ** M) * r0
print(f"After M={M} stages: Iterative residual = {r_iter:.8f} == Analytical (1-eta)^M r0 = {r_exact:.8f}")
assert np.isclose(r_iter, r_exact, rtol=1e-15)


Exercise 1: Geometric Residual Contraction r_M = (1 - eta)^M r_0 Verification
After M=20 stages: Iterative residual = 0.38759531 == Analytical (1-eta)^M r0 = 0.38759531


**Discussion.** Each stage subtracts a fraction $\eta$ of the remaining residual ($r_m = r_{m-1} - \eta r_{m-1} = (1 - \eta)r_{m-1}$), yielding exponential decay $r_M = (1-\eta)^M r_0 \approx e^{-\eta M} r_0$.

---

## Exercise 2

**Problem.** Examine a learning rate that is too large: train `GradientBoostingRegressor(n_estimators=15, learning_rate=2.10, max_depth=2)` on `HV_true` and compare its error with that of $\eta = 0.20$. For $\eta > 2$ each stage overshoots by more than the residual it corrects ($1 - \eta = -1.1$), so the residual changes sign and grows.

In [3]:
heading("Exercise 2: A learning rate that is too large (eta = 2.10 against eta = 0.20)")
gbm_stable = trees_ensembles.GradientBoostingRegressor(n_estimators=15, learning_rate=0.20, max_depth=2).fit(x_Ni, HV_true)
gbm_overshoot = trees_ensembles.GradientBoostingRegressor(n_estimators=15, learning_rate=2.10, max_depth=2).fit(x_Ni, HV_true)
mse_stable = float(np.mean((gbm_stable.predict(x_Ni) - HV_true) ** 2))
mse_overshoot = float(np.mean((gbm_overshoot.predict(x_Ni) - HV_true) ** 2))
print(f"15-Stage MSE: Stable shrinkage (eta=0.20) = {mse_stable:.4f} vs Divergent Overshoot (eta=2.10) = {mse_overshoot:.4f}")
assert mse_overshoot > 10.0 * mse_stable


Exercise 2: A learning rate that is too large (eta = 2.10 against eta = 0.20)
15-Stage MSE: Stable shrinkage (eta=0.20) = 1.9629 vs Divergent Overshoot (eta=2.10) = 3676.3016


**Discussion.** If every tree fitted its residual exactly, the residual after a stage would be $(1 - \eta)$ times the one before: it shrinks for $0 < \eta < 2$ (with alternating sign when $\eta > 1$) and grows for $\eta > 2$. With $\eta = 2.10$ the error after 15 stages is $3676\;\mathrm{HV}^2$, against $2.0\;\mathrm{HV}^2$ for $\eta = 0.20$. In practice the learning rate is kept well below one.

---

## Exercise 3

**Problem.** Implement an **early-stopping validation monitor** `gbm_early_stopping(X_tr, y_tr, X_val, y_val, max_stages=40, lr=0.15, max_depth=2)` that evaluates held-out validation MSE at every boosting stage $m = 1, \dots, M_{\max}$ and returns the stage $m^*$ minimizing validation error. Verify it on noisy Nordheim alloy train/validation data.

In [4]:
heading("Exercise 3: Implement Stagewise Early-Stopping Validation Monitor for GBM")
def gbm_early_stopping(
    X_tr: np.ndarray, y_tr: np.ndarray, X_val: np.ndarray, y_val: np.ndarray, max_stages: int = 40, lr: float = 0.15, max_depth: int = 2
) -> dict:
    """Track stagewise validation MSE across boosting iterations and select optimal stage m*."""
    f0 = float(np.mean(y_tr))
    p_tr = np.full(len(y_tr), f0)
    p_val = np.full(len(y_val), f0)
    val_mses = []
    for _ in range(max_stages):
        t_m = trees_ensembles.DecisionTreeRegressor(max_depth=max_depth).fit(X_tr, y_tr - p_tr)
        p_tr += lr * t_m.predict(X_tr)
        p_val += lr * t_m.predict(X_val)
        val_mses.append(float(np.mean((p_val - y_val) ** 2)))
    best_m = int(np.argmin(val_mses)) + 1
    return {"best_stage": best_m, "min_val_mse": val_mses[best_m - 1], "val_mses": val_mses}

es_res = gbm_early_stopping(x_Ni[::2], HV_noisy[::2], x_Ni[1::2], HV_true[1::2], max_stages=35, lr=0.20, max_depth=2)
print(f"Stage 1 Validation MSE = {es_res['val_mses'][0]:.3f} -> Optimal Stage m* = {es_res['best_stage']} (Val MSE = {es_res['min_val_mse']:.3f})")
assert es_res["min_val_mse"] < 0.15 * es_res["val_mses"][0] and es_res["best_stage"] >= 5


Exercise 3: Implement Stagewise Early-Stopping Validation Monitor for GBM
Stage 1 Validation MSE = 148.408 -> Optimal Stage m* = 14 (Val MSE = 2.953)


**Discussion.** The number of boosting stages $M$ directly controls model complexity; monitoring validation error per stage gives a stopping rule at no extra cost.